# Día 3 · Del panel a una decisión con Power BI

**Pregunta de cierre.** ¿Qué expedientes revisaríamos primero y cómo sabríamos si esa revisión aportó valor?

Vamos a reconstruir en **Power BI Desktop** el panel de los notebooks del día 2. Si el aula trabaja en navegador, hay una ruta equivalente en **Looker Studio**. Python prepara el archivo. El informe se construye en la herramienta de BI siguiendo los pasos de abajo.

## 1 · Preparar el CSV que vamos a importar

Primero ejecuta la exportación del [notebook 16](16_panel_siniestros_dia_2.ipynb). Aquí añadimos el mes y el orden de los días para que los gráficos queden ordenados. La medida es `expedientes`. Cada fila del CSV es un grupo de expedientes, así que los indicadores deben **sumar** esa columna.

In [ ]:
from pathlib import Path

import pandas as pd

archivo = 'resumen_siniestros_panel.csv'
inicio = Path.cwd()
carpetas = [inicio, inicio / 'data', inicio.parent / 'data', inicio / 'PySpark' / 'data']
ruta = next((p / archivo for p in carpetas if (p / archivo).is_file()), None)
if ruta is None:
    raise FileNotFoundError(f'Sube {archivo} o ejecuta la exportación del notebook 16.')

bi = pd.read_csv(ruta, parse_dates=['fecha'])
dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
bi['mes'] = bi['fecha'].dt.to_period('M').dt.to_timestamp()
bi['orden_dia'] = bi['fecha'].dt.dayofweek + 1
bi['dia_semana'] = bi['orden_dia'].map(dict(enumerate(dias, start=1)))
assert bi['expedientes'].sum() == 7721

# El ejemplo solo guarda un archivo local. En Colab se puede descargar desde Archivos.
destino = Path('siniestros_power_bi.csv')
bi.to_csv(destino, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
print(f'Archivo para BI: {destino.resolve()}')
bi.head()

## 2 · Construir el panel en Power BI Desktop

1. En **Inicio → Obtener datos → Texto/CSV**, abre `siniestros_power_bi.csv`. Comprueba la codificación UTF-8. En **Transformar datos**, revisa que `fecha` y `mes` sean fechas y que `hora`, `orden_dia` y `expedientes` sean números enteros. Carga la tabla y llámala `Siniestros`.
2. Crea una medida en **Modelado → Nueva medida**. Una medida vuelve a calcularse según los filtros de la página.

```dax
Total expedientes = SUM(Siniestros[expedientes])
```

3. Añade dos **segmentaciones** con `distrito` y `fecha`. Usa la opción **Entre** para el rango de fechas. Antes de montar más gráficos, comprueba con la tarjeta que ambos filtros funcionan.
4. Crea estas visualizaciones:

| Visual | Campos |
| --- | --- |
| Tarjeta | `Total expedientes` |
| Línea | Eje X: `mes` como fecha, sin jerarquía. Eje Y: `Total expedientes` |
| Barras | Categoría: `distrito` o `barrio`. Valor: `Total expedientes`. Ordena por el valor |
| Matriz de calor | Filas: `dia_semana`. Columnas: `hora`. Valores: `Total expedientes`. Aplica color de fondo mediante una escala |

5. Selecciona `dia_semana` en la vista de datos y usa **Ordenar por columna → orden_dia** para que el lunes aparezca primero. Cambia los filtros y comprueba que se aplican a todos los visuales.

**Comprobación.** Todo 2023 debe sumar **7.721 expedientes**. Eixample del 1 de enero al 30 de junio debe sumar **1.099**. Guarda el informe en el equipo del aula si quieres retomarlo.

[Importar CSV](https://learn.microsoft.com/en-us/power-query/connectors/text-csv) · [Segmentaciones](https://learn.microsoft.com/en-us/power-bi/visuals/power-bi-visualization-slicer-visual) · [Color en matrices](https://learn.microsoft.com/en-us/power-bi/desktop-conditional-table-formatting)

## 3 · Alternativa en navegador. Looker Studio

Importa el mismo CSV a una hoja de Google Sheets autorizada para la clase y conecta esa hoja en Looker Studio. Revisa los tipos de fecha y número. Usa **SUMA de expedientes** como métrica en una tarjeta, una serie temporal por mes y unas barras por distrito. Añade un control de periodo y un desplegable de distrito. Si quieres reproducir el mapa de calor, usa una tabla dinámica con día de la semana en filas y hora en columnas y aplica color a la métrica.

Comprueba los mismos dos totales antes de comparar las herramientas. Para la clase basta con trabajar en el informe privado de tu cuenta.

[Conectar Google Sheets](https://docs.cloud.google.com/looker/docs/studio/connect-to-google-sheets) · [Controles de Looker Studio](https://docs.cloud.google.com/looker/docs/studio/about-controls)

## 4 · La V de valor. Defender la siguiente decisión

Cada grupo muestra un filtro y una cifra. Después completa estas cuatro frases:

1. **Hallazgo.** «En ___, durante ___, se registraron ___ expedientes en la franja ___».
2. **Acción.** «Revisaríamos primero ___ porque ___».
3. **Límite.** «Para comparar el riesgo pediríamos ___». Puede ser volumen de tráfico por franja o un dato de exposición comparable.
4. **Cómo comprobar el valor.** «En el siguiente periodo mediríamos ___ para saber si la revisión ayudó». Distingue una métrica del proceso, como el tiempo hasta priorizar una revisión, de una métrica de resultado, como una tasa de siniestros ajustada por tráfico.

Si proponéis una intervención, comparad periodos de duración similar y buscad una zona de comparación antes de atribuirle un cambio. Cerrad con una decisión concreta que alguien pueda ejecutar.

[Volver al README](../README.md)